Proyecto final Unidad 6 Filtros

Universidad de Antioquia

William Andres Mora Herrera

Jean Carlos Suarez Hernandez

In [20]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal
import plotly.graph_objects as go

#Frecuencia de muestreo del dataset
FS = 200.0
Ruta = Path("Archivos caídas\F03_SA01_R05.txt")

#columnas con los 9 canales contenidos en los dataset
COLUMNAS = [
    "ADXL_X", "ADXL_Y", "ADXL_Z",
    "ITG_X",  "ITG_Y",  "ITG_Z",
    "MMA_X",  "MMA_Y",  "MMA_Z"
]

# Factores de conversión de bits (ADC) a unidades físicas (g para acelerómetros, °/s para giroscopio)
FACTORES = {
    "ADXL": (2 * 16) / (2**13), # Rango +-16g, resolución 13 bits
    "ITG":  (2 * 2000) / (2**16), # Rango +-2000°/s, resolución 16 bits
    "MMA":  (2 * 8) / (2**14) # Rango +-8g, resolución 14 bits
}

In [21]:
def cargar_senal(ruta=Ruta, magnitud=True):
    #leemos el archivo de texto y lo convertimos a un dataframe de pandas
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)
    #la ultima columna contiene un caracter de punto y coma al final, lo eliminamos y convertimos a int
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)

    # Escalamos los datos crudos a unidades reales usandos los factores de conversión antes declarados
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor

    #calculamos la magnitud de cada sensor para que los datos no dependan de la posicion del sensor, 
    #si magnitud es True, se calcula la magnitud de cada sensor y se devuelve un dataframe con las magnitudes
    if magnitud:
        df = pd.DataFrame({
            prefijo: np.sqrt(df[f'{prefijo}_X']**2 + df[f'{prefijo}_Y']**2 + df[f'{prefijo}_Z']**2)
            for prefijo in FACTORES
        })
        
    return df

df_senal = cargar_senal()
print(f"Archivo cargado: {Ruta.name} | Filas: {len(df_senal)}")
print(df_senal.head())

Archivo cargado: F03_SA01_R05.txt | Filas: 3000
       ADXL        ITG       MMA
0  0.897639  52.104544  0.731316
1  0.837096  57.194338  0.691062
2  0.777815  60.408526  0.650066
3  0.739881  60.809458  0.623442
4  0.754887  58.595626  0.637319


In [22]:
# Especificaciones del filtro pasa-altas para eliminar la gravedad (DC) y movimientos lentos
FRECHAZO = 6.0   # Frecuencia de parada (Hz)
FPASO = 11.0     # Frecuencia de paso (Hz)
GPASS = 1.0      # Rizado máximo en banda de paso (dB)
GSTOP = 25.0     # Atenuación mínima en banda de rechazo (dB)

nyq = FS / 2
# Calculamos el orden mínimo y la frecuencia de corte normalizada
orden_hpf, wn_hpf = signal.buttord(FPASO / nyq, FRECHAZO / nyq, GPASS, GSTOP)
# Usamos formato SOS (Second-Order Sections) por estabilidad numérica en orden 6
sos_hpf = signal.butter(orden_hpf, wn_hpf, btype='highpass', output='sos')

def filtrar_senal(df, columna='ADXL'):
    # Aplicamos filtfilt (ida y vuelta) para tener fase cero y no desfasar los picos del impacto
    return pd.Series(signal.sosfiltfilt(sos_hpf, df[columna].values), index=df.index)

senal_filtrada = filtrar_senal(df_senal, columna='ADXL')
print(f"Orden: {orden_hpf} | Pico filtrado: {senal_filtrada.abs().max():.3f} g")

Orden: 6 | Pico filtrado: 0.409 g


In [23]:
from plotly.subplots import make_subplots

def graficar_bode():
    # Calculamos la respuesta en frecuencia del filtro digital (w en Hz, h respuesta compleja)
    w, h = signal.sosfreqz(sos_hpf, worN=2048, fs=FS)
    
    # Convertimos la magnitud a decibelios (dB); usamos np.maximum para evitar log(0)
    mag_db = 20 * np.log10(np.maximum(np.abs(h), 1e-12))
    # Al aplicar filtfilt (ida y vuelta), la atenuación en dB se duplica
    mag_eff_db = 2 * mag_db
    # Extraemos la fase continua desenrollada y la pasamos de radianes a grados
    fase_deg = np.unwrap(np.angle(h)) * (180 / np.pi)
    # Desnormalizamos la frecuencia de corte Wn a Hz
    fc_hz = wn_hpf * (FS / 2)

    # Creamos una figura con 2 subgráficas apiladas que comparten el eje X (frecuencia)
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=(
            f"Diagrama de Bode: Butter highpass, orden {orden_hpf}, Wn = {fc_hz:.1f} Hz",
            "Fase"
        )
    )

    # Subgráfica 1: Magnitud de un solo paso vs. magnitud efectiva de fase cero
    fig.add_trace(go.Scatter(x=w, y=mag_db, name="Un paso (freqz)", line=dict(color="#1f77b4")), row=1, col=1)
    fig.add_trace(go.Scatter(x=w, y=mag_eff_db, name="Efectiva (ida y vuelta, filtfilt)", line=dict(color="red", dash="dash")), row=1, col=1)
    
    # Líneas verticales de referencia para las especificaciones del filtro (Wn, fs y fp)
    fig.add_vline(x=fc_hz, line_dash="dash", line_color="gray", annotation_text=f"Wn={fc_hz:.1f} Hz", row=1, col=1)
    fig.add_vline(x=FRECHAZO, line_dash="dot", line_color="gray", annotation_text=f"fs={FRECHAZO} Hz", row=1, col=1)
    fig.add_vline(x=FPASO, line_dash="dot", line_color="gray", annotation_text=f"fp={FPASO} Hz", row=1, col=1)

    # Subgráfica 2: Fase introducida en un paso vs. fase nula (0°) lograda con filtfilt
    fig.add_trace(go.Scatter(x=w, y=fase_deg, name="Fase un paso", line=dict(color="#1f77b4"), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=w, y=np.zeros_like(w), name="Fase efectiva (0°)", line=dict(color="red", dash="dash"), showlegend=False), row=2, col=1)

    # Configuramos escala logarítmica en frecuencia (hasta Nyquist) y límites de los ejes Y
    fig.update_xaxes(type="log", range=[np.log10(0.1), np.log10(FS / 2)], title_text="Frecuencia (Hz)", row=2, col=1)
    fig.update_yaxes(range=[-100, 5], title_text="Magnitud (dB)", row=1, col=1)
    fig.update_yaxes(title_text="Fase (°)", row=2, col=1)
    fig.update_layout(height=550, legend=dict(orientation="h", y=1.12))
    fig.show()

def graficar_senal(df=df_senal, filtrada=senal_filtrada, ruta=Ruta, columna='ADXL'):
    # Identificamos si el archivo es actividad diaria (D) o caída (F) según su nombre
    clase = "ADL" if ruta.name.startswith("D") else "Caída"
    # Construimos el vector de tiempo en segundos a partir de la frecuencia de muestreo
    t = np.arange(len(df)) / FS

    # Superponemos la señal original (con gravedad) y la señal filtrada (solo altas frecuencias)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t, y=df[columna], name="Original", line=dict(color="gray", width=1)))
    fig.add_trace(go.Scatter(x=t, y=filtrada, name="Filtrada", line=dict(color="#1f77b4", width=1.2)))
    fig.update_layout(
        title=f"{ruta.stem} ({clase}) — Pico filtrado: {filtrada.abs().max():.3f} g",
        xaxis_title="Tiempo (s)",
        yaxis_title="Aceleración (g)",
        height=350,
        hovermode="x unified"
    )
    fig.show()

# Ejecutamos ambas visualizaciones
graficar_bode()
graficar_senal()

In [24]:
# Umbral de decisión en gravedad (g) calibrado para separar impactos bruscos de actividades diarias (como trotar)
UMBRAL_ADXL = 1.75

def detectar_caida(senal_filtrada, umbral=UMBRAL_ADXL):
    # Extraemos la amplitud máxima absoluta de la señal tras eliminar bajas frecuencias
    pico = float(senal_filtrada.abs().max())
    
    # Regla de clasificación binaria: si el impacto supera el umbral, se marca como caída
    es_caida = pico > umbral
    etiqueta = "Caída" if es_caida else "ADL"
    
    return es_caida, etiqueta, pico

# Obtenemos la etiqueta real a partir de la nomenclatura del dataset SisFall ('D' = ADL, 'F' = Fall)
clase_real = "ADL" if Ruta.name.startswith("D") else "Caída"

# Evaluamos la señal filtrada actual con nuestro clasificador
es_caida, etiqueta, pico = detectar_caida(senal_filtrada)

# Imprimimos el resumen comparando la predicción del sistema contra la clase real
print(f"Archivo:   {Ruta.name}")
print(f"Predicho:  {etiqueta} (Pico: {pico:.4f} g | Umbral: {UMBRAL_ADXL} g)")
print(f"Real:      {clase_real}")

Archivo:   F03_SA01_R05.txt
Predicho:  ADL (Pico: 0.4090 g | Umbral: 1.75 g)
Real:      Caída


In [25]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.signal as signal

# Aseguramos que la carpeta apunte al directorio padre de tu ruta de ejemplo
CARPETA = Ruta.parent 

def extraer_picos_dataset(carpeta=CARPETA, columna='ADXL'):
    """
    Recorre todos los archivos .txt de la carpeta, aplica el filtro sos_lpf ya diseñado
    y extrae el pico máximo absoluto (en g) junto con su clase real.
    """
    registros = []
    for ruta_txt in sorted(glob.glob(str(Path(carpeta) / "*.txt"))):
        ruta_obj = Path(ruta_txt)
        nombre = ruta_obj.stem
        
        # Ignoramos archivos auxiliares que no empiecen por D (ADL) o F (Caída)
        if not (nombre.startswith("D") or nombre.startswith("F")):
            continue
            
        df = cargar_senal(ruta=ruta_obj, magnitud=True)
        senal_filt = filtrar_senal(df, columna=columna)
        
        clase_bin = 1 if nombre.startswith("F") else 0
        registros.append({
            'Archivo': nombre,
            'Clase': 'Caída' if clase_bin == 1 else 'ADL',
            'y': clase_bin,
            'Pico_g': float(senal_filt.abs().max())
        })
        
    return pd.DataFrame(registros)


def evaluar_loocv(df_datos, nombre_escenario=""):
    """
    Ejecuta Leave-One-Out Cross-Validation optimizando la Exactitud Balanceada
    (promedio entre Sensibilidad y Especificidad) para evitar sesgo por desbalance de clases.
    """
    df = df_datos.reset_index(drop=True)
    # Grilla fina de 500 umbrales candidatos entre el mínimo y máximo pico del dataset
    candidatos = np.linspace(df['Pico_g'].min(), df['Pico_g'].max(), 500)
    
    umbrales_fold = []
    predicciones = []
    
    # Iteramos dejando 1 archivo por fuera en cada paso (N-1 entrenamiento, 1 prueba)
    for i in range(len(df)):
        train = df.drop(index=i)
        picos_caida = train[train['y'] == 1]['Pico_g'].values
        picos_adl   = train[train['y'] == 0]['Pico_g'].values
        
        # Buscamos el umbral óptimo en el conjunto de entrenamiento (N-1)
        mejor_u = max(
            candidatos,
            key=lambda u: 0.5 * ((picos_caida >= u).mean() + (picos_adl < u).mean())
        )
        
        umbrales_fold.append(mejor_u)
        predicciones.append(int(df.loc[i, 'Pico_g'] >= mejor_u))
        
    df['Umbral_LOO_g'] = umbrales_fold
    df['Pred_y'] = predicciones
    df['Acierto'] = df['y'] == df['Pred_y']
    
    # Cálculo de métricas de rendimiento
    caidas = df[df['y'] == 1]
    adls   = df[df['y'] == 0]
    
    sensibilidad  = caidas['Acierto'].mean() * 100
    especificidad = adls['Acierto'].mean() * 100
    acc_global    = df['Acierto'].mean() * 100
    acc_balance   = (sensibilidad + especificidad) / 2
    
    resumen = {
        'Escenario': nombre_escenario,
        'Total Archivos': len(df),
        'Umbral LPF Prom. (g)': round(np.mean(umbrales_fold), 4),
        'Umbral Std (g)': round(np.std(umbrales_fold), 4),
        'Sensibilidad (%)': round(sensibilidad, 2),
        'Especificidad (%)': round(especificidad, 2),
        'Exactitud Balanceada (%)': round(acc_balance, 2),
        'Exactitud Global (%)': round(acc_global, 2),
        'Caídas Detectadas': f"{caidas['Acierto'].sum()}/{len(caidas)}"
    }
    
    return df, resumen


def mostrar_errores(df_resultados, titulo):
    """
    Filtra e imprime las señales mal clasificadas diferenciando Falsos Negativos y Falsos Positivos.
    """
    errores = df_resultados[~df_resultados['Acierto']].copy()
    print(f"\n{'='*70}")
    print(f"❌ ERRORES DE CLASIFICACIÓN — {titulo} (Total: {len(errores)})")
    print(f"{'='*70}")
    
    if errores.empty:
        print("¡Clasificación perfecta! No hubo señales mal clasificadas en este escenario.")
        return
        
    errores['Tipo_Error'] = np.where(
        errores['y'] == 1,
        'Falso Negativo (Caída no detectada)',
        'Falso Positivo (ADL marcada como Caída)'
    )
    errores['Diferencia_Umbral_g'] = (errores['Pico_g'] - errores['Umbral_LOO_g']).round(4)
    
    cols_mostrar = ['Archivo', 'Clase', 'Tipo_Error', 'Pico_g', 'Umbral_LOO_g', 'Diferencia_Umbral_g']
    display(errores[cols_mostrar].sort_values(by=['Tipo_Error', 'Archivo']))


# ==============================================================================
# EJECUCIÓN DE LA PRUEBA DE HIPÓTESIS CON LPF (CON VS. SIN SEÑAL ANÓMALA)
# ==============================================================================

# 1. Extraemos los picos filtrados con LPF de toda la carpeta
df_completo = extraer_picos_dataset(CARPETA, columna='ADXL')

# Verificamos el valor exacto del archivo anómalo D04_SA03_R01
fila_anomala = df_completo[df_completo['Archivo'] == 'D04_SA03_R01']
if not fila_anomala.empty:
    print(f"🔎 Pico filtrado LPF de D04_SA03_R01: {fila_anomala['Pico_g'].values[0]:.4f} g\n")
else:
    print("⚠️ Advertencia: No se encontró 'D04_SA03_R01' en la carpeta.\n")

# 2. Escenario A: Dataset completo (incluyendo D04_SA03_R01)
df_res_con, metricas_con = evaluar_loocv(df_completo, "1. Con D04_SA03_R01 (LPF)")

# 3. Escenario B: Dataset depurado (descartando D04_SA03_R01)
df_sin_anomalo = df_completo[df_completo['Archivo'] != 'D04_SA03_R01'].copy()
df_res_sin, metricas_sin = evaluar_loocv(df_sin_anomalo, "2. Sin D04_SA03_R01 (LPF)")

# 4. Tabla comparativa de resultados
df_comparacion = pd.DataFrame([metricas_con, metricas_sin]).set_index('Escenario')
display(df_comparacion)

# 5. Identificar qué caídas leves se rescatan al eliminar la anomalía
caidas_ganadas = set(df_res_sin[(df_res_sin['y'] == 1) & (df_res_sin['Acierto'])]['Archivo']) - \
                 set(df_res_con[(df_res_con['y'] == 1) & (df_res_con['Acierto'])]['Archivo'])

print(f"\n✔ Caídas leves rescatadas al descartar D04_SA03_R01 ({len(caidas_ganadas)}):")
if caidas_ganadas:
    display(df_res_sin[df_res_sin['Archivo'].isin(caidas_ganadas)][['Archivo', 'Pico_g', 'Umbral_LOO_g']])
else:
    print("Ninguna caída adicional superó el nuevo umbral.")

# 6. Listado detallado de las señales en las que se equivocó cada escenario
mostrar_errores(df_res_con, "Escenario 1: Con D04_SA03_R01")
mostrar_errores(df_res_sin, "Escenario 2: Sin D04_SA03_R01")

🔎 Pico filtrado LPF de D04_SA03_R01: 1.7222 g



,Total Archivos,Umbral LPF Prom. (g),Umbral Std (g),Sensibilidad (%),Especificidad (%),Exactitud Balanceada (%),Exactitud Global (%),Caídas Detectadas
Escenario,,,,,,,,
1. Con D04_SA03_R01 (LPF),68,1.7424,0.0369,55.00,87.50,71.25,58.82,33/60
2. Sin D04_SA03_R01 (LPF),67,1.4339,0.0511,58.33,85.71,72.02,61.19,35/60



✔ Caídas leves rescatadas al descartar D04_SA03_R01 (2):


,Archivo,Pico_g,Umbral_LOO_g
19,F02_SA01_R03,1.679991,1.440214
37,F04_SA01_R01,1.491263,1.440214



❌ ERRORES DE CLASIFICACIÓN — Escenario 1: Con D04_SA03_R01 (Total: 28)


,Archivo,Clase,Tipo_Error,Pico_g,Umbral_LOO_g,Diferencia_Umbral_g
18,F02_SA01_R01,Caída,Falso Negativo (Caída no detectada),0.805742,1.746956,-0.9412
20,F02_SA01_R03,Caída,Falso Negativo (Caída no detectada),1.679991,1.746956,-0.0670
24,F02_SA03_R02,Caída,Falso Negativo (Caída no detectada),0.653366,1.746956,-1.0936
25,F02_SA03_R03,Caída,Falso Negativo (Caída no detectada),0.927645,1.746956,-0.8193
26,F02_SA03_R04,Caída,Falso Negativo (Caída no detectada),0.881563,1.746956,-0.8654
27,F02_SA03_R05,Caída,Falso Negativo (Caída no detectada),0.508875,1.746956,-1.2381
28,F03_SA01_R01,Caída,Falso Negativo (Caída no detectada),0.460550,1.746956,-1.2864
29,F03_SA01_R02,Caída,Falso Negativo (Caída no detectada),1.312143,1.746956,-0.4348
30,F03_SA01_R03,Caída,Falso Negativo (Caída no detectada),1.180484,1.746956,-0.5665
31,F03_SA01_R04,Caída,Falso Negativo (Caída no detectada),0.503001,1.746956,-1.2440



❌ ERRORES DE CLASIFICACIÓN — Escenario 2: Sin D04_SA03_R01 (Total: 26)


,Archivo,Clase,Tipo_Error,Pico_g,Umbral_LOO_g,Diferencia_Umbral_g
17,F02_SA01_R01,Caída,Falso Negativo (Caída no detectada),0.805742,1.440214,-0.6345
23,F02_SA03_R02,Caída,Falso Negativo (Caída no detectada),0.653366,1.440214,-0.7868
24,F02_SA03_R03,Caída,Falso Negativo (Caída no detectada),0.927645,1.440214,-0.5126
25,F02_SA03_R04,Caída,Falso Negativo (Caída no detectada),0.881563,1.440214,-0.5587
26,F02_SA03_R05,Caída,Falso Negativo (Caída no detectada),0.508875,1.440214,-0.9313
27,F03_SA01_R01,Caída,Falso Negativo (Caída no detectada),0.460550,1.440214,-0.9797
28,F03_SA01_R02,Caída,Falso Negativo (Caída no detectada),1.312143,1.440214,-0.1281
29,F03_SA01_R03,Caída,Falso Negativo (Caída no detectada),1.180484,1.440214,-0.2597
30,F03_SA01_R04,Caída,Falso Negativo (Caída no detectada),0.503001,1.440214,-0.9372
31,F03_SA01_R05,Caída,Falso Negativo (Caída no detectada),0.408954,1.440214,-1.0313
